In [6]:
from ta.volatility import AverageTrueRange

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from backtesting import Strategy
from backtesting import Backtest


from datetime import timedelta
import Util
import importlib
importlib.reload(Util)

from enum import Enum
import time


In [7]:
class KeyLevel(Enum):
    SUPPORT = 1
    RESISTANT = 2

In [8]:
pair = 'GBPJPY'

df_d1_raw = pd.read_csv(f"../Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h4_raw = pd.read_csv(f"../Trading Data/{pair}_H4.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_m15_raw = pd.read_csv(f"../Trading Data/{pair}_M15.csv",sep=',', parse_dates=['Date'], index_col='Date')

print(df_d1_raw)
print(df_h4_raw)
print(df_m15_raw)

               Open     High      Low    Close  Tickvol
Date                                                   
2010-09-10  129.605  130.065  128.995  129.255   172229
2010-09-13  129.565  130.115  128.780  129.090   215859
2010-09-14  129.105  129.430  127.640  129.015   226170
2010-09-15  129.015  134.070  128.555  133.975   252317
2010-09-16  133.975  134.260  132.970  134.150   222443
...             ...      ...      ...      ...      ...
2026-09-18  208.323  211.276  208.233  210.095   364707
2026-09-21  210.296  210.762  209.676  210.321   243731
2026-09-22  210.321  210.892  209.513  210.080   333200
2026-09-23  210.080  210.176  209.422  209.564   273936
2026-09-24  209.566  210.105  208.768  209.848   199988

[4183 rows x 5 columns]
                        Open     High      Low    Close  Tickvol
Date                                                            
2010-09-10 04:00:00  129.605  129.630  128.995  129.385    25539
2010-09-10 08:00:00  129.380  129.985  129.275  129.

In [9]:
structure_d1_window = 2
structure_m15_window = 2

date_format = '%Y-%m-%d %H:%M'
# start_date = '2023-1-1 00:00'
# end_date = '2024-3-29 00:00'
start_date = datetime.strptime('2021-1-1 00:00', date_format)
end_date = datetime.strptime('2026-9-16 00:00', date_format)
# start_date = '2022-01-1 00:00'
# end_date = '2026-9-16 00:00'

timezone = "Europe/London"

# nombre de jours pour les high.low
define_key_level_window = 90

# gap t avoid look ahead
gap_look_ahead = 6

# nombre de candle au dessus/ en desous de key level après breakout
breakout_candles = 9

# nombre de jours à chercher key level
key_level_lookback_candle = 100

# vérifier si 30 candle dans ke passé pour valider un valid breakout,
# un valid breakout doit avoir {breakout_candles} candles au dessus du resistant level
breakout_candles_lookback = 30

start_trade_hour = 9
stop_trade_hour = 19

atr_multiplier = 0.7
min_touched = 3

In [10]:
# process D1 data
df_d1 = df_d1_raw.loc[start_date - timedelta(days=30) : end_date]

df_d1 = Util.implement_market_structure_states_fixed_pivots(df_d1, n=structure_d1_window)
df_d1["state_changed"] = df_d1["market_state"] != df_d1[
    "market_state"
].shift(1)
print(df_d1[df_d1["state_changed"]][["Close", "market_state"]])
df_d1 = df_d1.loc[start_date : end_date]

              Close market_state
Date                            
2020-12-02  139.632          NaN
2020-12-03  139.680          NaN
2020-12-04  139.937          NaN
2020-12-07  139.114          NaN
2020-12-08  139.134          NaN
...             ...          ...
2026-04-30  213.107      BEARISH
2026-05-25  214.592      BULLISH
2026-06-18  212.961      BEARISH
2026-06-29  214.706      BULLISH
2026-07-30  215.005      BEARISH

[82 rows x 2 columns]


In [11]:
# process h4 data
df_h4 = df_h4_raw.loc[start_date - timedelta(days=20) : end_date]
df_h4 = Util.implement_market_structure_states_fixed_pivots(df_h4, 5)
df_h4 = df_h4.loc[start_date : end_date]

h4_d1_direction_list = []
for i in range(len(df_d1)):
    row = df_d1.iloc[i]
    start_date_daily = row.name
    end_date_daily = start_date_daily + timedelta(hours=23, minutes=59)
    df_by_day = df_h4.loc[f'{start_date_daily}' : f'{end_date_daily}']

    for index2, row2 in df_by_day.iterrows():
        h4_d1_direction_list.append(df_d1.iloc[i-1]['market_state'])

df_h4['market_state_d1'] = h4_d1_direction_list

indicator_atr_14 = AverageTrueRange(df_h4['High'], df_h4['Low'], df_h4['Close'], window=14)
df_h4['ATR'] = indicator_atr_14._atr

df_h4

,Open,High,Low,Close,Tickvol,is_swing_high,is_swing_low,market_state,market_state_d1,ATR
Date,,,,,,,,,,
2021-01-04 00:00:00,140.944,141.304,140.776,141.005,37352,True,False,BULLISH,BEARISH,0.000000
2021-01-04 04:00:00,141.007,141.043,140.847,140.938,20550,False,False,BULLISH,BEARISH,0.000000
2021-01-04 08:00:00,140.936,141.064,140.412,140.757,47343,False,False,BULLISH,BEARISH,0.000000
2021-01-04 12:00:00,140.758,140.861,140.188,140.226,55445,False,False,BULLISH,BEARISH,0.000000
2021-01-04 16:00:00,140.227,140.233,139.687,139.870,55631,False,False,BULLISH,BEARISH,0.000000
...,...,...,...,...,...,...,...,...,...,...
2026-09-15 08:00:00,208.652,209.090,208.514,208.655,56407,False,False,BEARISH,BEARISH,0.567701
2026-09-15 12:00:00,208.655,209.211,208.607,209.168,74480,False,False,BULLISH,BEARISH,0.570294
2026-09-15 16:00:00,209.167,209.224,208.885,209.066,38410,False,False,BULLISH,BEARISH,0.553773


In [12]:
# process m15 data
def process_m15_data(df_m15, df_h4):
    df_m15 = Util.implement_market_structure_states_fixed_pivots(df_m15, n=structure_m15_window)

    m15_daily_direction_list = []

    m15_h4_resistant_level_list = []
    m15_is_resistants_list = []
    m15_is_below_resistant_list = []

    m15_h4_support_level_list = []
    m15_is_supports_list = []
    m15_is_above_support_list = []

    index_list = []

    for i in range(len(df_h4)):
        row = df_h4.iloc[i]
        start_date_h4 = row.name
        end_date_h4 = start_date_h4 + timedelta(hours=3, minutes=59)
        df_by_day = df_m15.loc[f'{start_date_h4}' : f'{end_date_h4}']

        for index2, row2 in df_by_day.iterrows():
            m15_daily_direction_list.append(df_h4.iloc[i]['market_state_d1'])

            m15_h4_resistant_level_list.append(df_h4.iloc[i]['resistant_point'])
            m15_is_resistants_list.append(df_h4.iloc[i]['is_resistant'])
            m15_is_below_resistant_list.append(df_h4.iloc[i]['valid_breakout_resistant_levels'])
            # affect current market states of H4 candle because the result is market state of previous D1 candle

            m15_h4_support_level_list.append(df_h4.iloc[i]['support_point'])
            m15_is_supports_list.append(df_h4.iloc[i]['is_support'])
            m15_is_above_support_list.append(df_h4.iloc[i]['valid_breakout_support_levels'])

            # ajust to daylight saving time
            dst = pd.Timestamp(index2, tz=timezone).dst()
            if dst == timedelta(0):
                index2 = index2 - timedelta(hours=1)
                index_list.append(index2)
            else:
                index_list.append(index2)

    df_m15['market_state_d1'] = m15_daily_direction_list

    df_m15['resistant_point'] = m15_h4_resistant_level_list
    df_m15['is_resistant'] = m15_is_resistants_list
    df_m15['valid_breakout_resistant_levels'] = m15_is_below_resistant_list

    df_m15['support_point'] = m15_h4_support_level_list
    df_m15['is_support'] = m15_is_supports_list
    df_m15['valid_breakout_support_levels'] = m15_is_above_support_list

    df_m15.index = index_list

    return df_m15

In [13]:
def is_existed(list, value):
    for i in range(len(list)):
        if list[i] == value:
            return True
    return False

In [14]:
# using m15 market structure for entry
def apply_total_signal(df, key_level_lookback_candle = 150, gap=1, breakout_candles_lookback = 30, start_trade_hour=8, stop_trade_hour=19):
    # Initialize the 'TotalSignal' column
    df_signal_list = []
    valid_support_levels_list = []
    valid_resistant_levels_list = []
    current_day = None

    # short variables
    is_resistant_level_touched = False
    is_below_resistant_levels = False
    high_tf_condition_short_satisfied = False
    high_tf_condition_short_candle = None
    m15_market_state_bearish = False

    # long variables
    is_support_level_touched = False
    is_above_support_levels = False
    high_tf_condition_long_satisfied = False
    high_tf_condition_long_candle = None
    m15_market_state_bullish = False

    # we want market structure to switch from bearish to bullish when htf conditions are met
    # if market structure is already bullish we need to wait
    previous_market_structure = None

    resistants_levels_touched = []
    supports_levels_touched = []

    for i in range(0, len(df)):
        current_bar_date = df.index[i]
        signal = 0
        # valid resistants levels are when valid breakout and levels touched
        valid_resistant_levels = []
        valid_support_levels = []

        if current_day != current_bar_date.date() or current_bar_date.time().hour >= stop_trade_hour:
            current_day = current_bar_date.date()
            # short
            high_tf_condition_short_satisfied = False
            resistants_levels_touched = []
            is_resistant_level_touched = False
            valid_breakout_resistant_levels = []
            is_below_resistant_levels = False

            # long
            high_tf_condition_long_satisfied = False
            supports_levels_touched = []
            is_support_level_touched = False
            valid_breakout_support_levels = []
            is_above_support_levels = False

        # short
        # d1 trending conditions
        c_short_daily_direction = df['market_state_d1'].iloc[i-1] == 'BEARISH'
        if c_short_daily_direction:
            c_m15_ms_bearish = df['market_state'].iloc[i] == 'BEARISH'

            # h4 key level conditions
            resistants_levels = Util.get_resistant_list(df, i, key_level_lookback_candle, gap, Util.TimeFrame.M15)
            for level in resistants_levels:
                if df['High'].iloc[i-1] > level:
                    is_resistant_level_touched = True
                    if not is_existed(resistants_levels_touched, level):
                        resistants_levels_touched.append(level)
            
            # Combine conditions
            if (is_resistant_level_touched and
                not high_tf_condition_short_satisfied):
                high_tf_condition_short_satisfied = True
                high_tf_condition_short_candle = current_bar_date
                previous_market_structure = df['market_state'].iloc[i]
                print(f"{current_bar_date} {i}")

            if (high_tf_condition_short_satisfied == True and
                c_m15_ms_bearish and
                high_tf_condition_short_candle < current_bar_date and
                previous_market_structure == 'BULLISH'):
                m15_market_state_bearish = True

            # breakout_candles_below candle needs to be below resistant level for the breakout to be valid
            valid_breakout_resistant_levels = Util.get_valid_breakout_resistant_level(df, i, breakout_candles_lookback)
            
            for valid_level in valid_breakout_resistant_levels:
                for level_touched in resistants_levels_touched:
                    if valid_level == level_touched:
                        valid_resistant_levels.append(valid_level)
                        is_below_resistant_levels = True

            c_time_trade = start_trade_hour <= current_bar_date.hour and current_bar_date.hour <= stop_trade_hour
            if c_time_trade:
                if (high_tf_condition_short_satisfied == True and
                    m15_market_state_bearish and
                    is_below_resistant_levels):
                    signal = 1

        # long
        c_long_daily_direction = df['market_state_d1'].iloc[i-1] == 'BULLISH'
        if c_long_daily_direction:
            c_m15_ms_bullish = df['market_state'].iloc[i] == 'BULLISH'
            
            # h4 support level conditions
            supports_levels = Util.get_support_list(df, i, key_level_lookback_candle, gap, Util.TimeFrame.M15)
            for level in supports_levels:
                if df['Low'].iloc[i-1] < level:
                    is_support_level_touched = True
                    if not is_existed(supports_levels_touched, level):
                        supports_levels_touched.append(level)

            # Combine conditions
            if (is_support_level_touched and
                not high_tf_condition_long_satisfied):
                high_tf_condition_long_satisfied = True
                m15_market_state_bullish = False
                high_tf_condition_long_candle = current_bar_date
                previous_market_structure = df['market_state'].iloc[i]
                # print(f"high_tf_condition_long_satisfied {current_bar_date} {i}")
    
            if (high_tf_condition_long_satisfied and
                c_m15_ms_bullish and
                high_tf_condition_long_candle < current_bar_date and
                previous_market_structure == 'BEARISH'):
                # print(f"m15_market_state_bullish {current_bar_date} {i}")
                m15_market_state_bullish = True

            # breakout_candles_above candle needs to be above support level for the breakout to be valid
            valid_breakout_support_levels = Util.get_valid_breakout_support_level(df, i, breakout_candles_lookback)
            for valid_level in valid_breakout_support_levels:
                for level_touched in supports_levels_touched:
                    if valid_level == level_touched:
                        valid_support_levels.append(valid_level)
                        is_above_support_levels = True

            c_time_trade = start_trade_hour <= current_bar_date.hour and current_bar_date.hour <= stop_trade_hour
            if c_time_trade:
                if (high_tf_condition_long_satisfied == True and
                    m15_market_state_bullish and
                    is_above_support_levels):
                    print(f"signal {current_bar_date} {i}")
                    signal = 2

        if previous_market_structure != df['market_state'].iloc[i]:
            if df['market_state'].iloc[i] == 'BULLISH':
                m15_market_state_bearish = False
            if df['market_state'].iloc[i] == 'BEARISH':
                m15_market_state_bullish = False
            previous_market_structure = df['market_state'].iloc[i]    

        df_signal_list.append(signal)
        valid_resistant_levels_list.append(valid_resistant_levels)
        valid_support_levels_list.append(valid_support_levels)

    return df_signal_list, valid_resistant_levels_list, valid_support_levels_list

In [15]:
class MyStrat(Strategy):
    mysize = 0.5
    slcoef = 2
    TPcoef = 2
    risk_percent = 0.01
    last_swing_low = None
    last_swing_high = None
    previous_swing_low = None
    previous_swing_high = None
    key_level_gap = 0.2
    last_trade_index = 0
    dfopt = None

    stoploss_info_list = []
    valid_resistant_level = None
    valid_support_level = None

    setup_duration_gap = 60
    stoploss_window = 150
    
    def init(self):
        self.current_day = None
        self.order_placed_time = None
        self.setup_duration = None
        
        self.stoploss_info_list = []
        self.valid_resistant_level = None
        self.valid_support_level = None

        super().init()

    def reset_order(self):
        # delete placed order at the end of the day if not executed
        if self.order_placed_time:
            for order in self.orders:
                order_executed= False
                for trade in self.trades:
                    if trade.tag == order.tag:
                        order_executed = True

                if order_executed == False:
                    order.cancel()

        self.order_placed_time = None
        self.setup_duration = None
        self.valid_resistant_level = None
        self.valid_support_level = None

    def get_highest_high(self):
        return self.data.High[len(self.data.High) - self.stoploss_window : len(self.data.High)].max()

    def get_lowest_low(self):
        return self.data.Low[len(self.data.Low) - self.stoploss_window  : len(self.data.Low)].min()

    def is_entry_in_valid_short_zone(self, valid_resistant_list):
        return self.data.Close[-1] < max(valid_resistant_list) + self.key_level_gap

    def is_entry_in_valid_long_zone(self, valid_support_list):
        return self.data.Close[-1] > min(valid_support_list) - self.key_level_gap

    def add_stoploss_info_list(self, info):
        is_added = False
        for stoploss_info in self.stoploss_info_list:
            if stoploss_info[0] == info[0] and stoploss_info[1] == info[1]:
                is_added = True
                stoploss_info[2] = info[2]

        if is_added == False:
            self.stoploss_info_list.append(info)

    def get_stoploss_info(self, key_level_type, key_level_value):
        for stoploss_info in self.stoploss_info_list:
            if stoploss_info[0] == key_level_type and stoploss_info[1] == key_level_value:
                return stoploss_info
        return None

    def dynamic_stoploss(self):
        # delete placed order at the end of the day if not executed
        if self.order_placed_time:
            for order in self.orders:
                order_executed= False
                for trade in self.trades:
                    if trade.tag == order.tag:
                        order_executed = True
                        if order.stop is not None:
                            # in long trade, short order is for sl and inverse for short trade
                            # so we identify short order for long trade to identify sl
                            if order.is_short:
                                self.add_stoploss_info_list([KeyLevel.SUPPORT, self.valid_support_level, order.stop])
                            else:
                                self.add_stoploss_info_list([KeyLevel.RESISTANT, self.valid_resistant_level, order.stop])


                if order_executed == False:
                    order.cancel()
                    if order.is_short:
                        sl = abs(order.stop - self.get_highest_high())
                        tp = order.stop - self.TPcoef * sl
                        position_size = int((self.risk_percent * self.equity) / sl)
                        self.sell(sl = self.get_highest_high(), tp = tp, size=position_size, stop=order.stop)
                    else:
                        sl = abs(order.stop - self.get_lowest_low())
                        tp = order.stop + self.TPcoef * sl
                        position_size = int((self.risk_percent * self.equity) / sl)
                        order = self.buy(sl = self.get_lowest_low(), tp = tp, size=position_size, stop=order.stop)

    def get_tag(self):
        return f"{self.order_placed_time}"

    def wait_3_candles_before_new_trade(self):
        return self.last_trade_index + 3
    
    def next(self):
        super().next()
        index = len(self.data.index)

        # when an order is placed, it will be valable for 12h
        if self.setup_duration is not None and index >= self.setup_duration:
            self.reset_order()

        # place sl to the lowest/highest
        self.dynamic_stoploss()

        if len(self.trades) > 0:
            self.last_trade_index = index

        if len(self.orders)==0 and index >= self.wait_3_candles_before_new_trade():
            if self.data['TotalSignal'][-1]==1:
                valid_resistant_list = Util.get_valid_resistant_level(self.dfopt, index, breakout_candles_lookback)
                if len(valid_resistant_list) > 0:
                    valid_resistant_level = max(valid_resistant_list)
                    is_entry_in_valid_zone = self.is_entry_in_valid_short_zone(valid_resistant_list)
                    planned_entry_price = 0
                    sl = 0

                    if is_entry_in_valid_zone: 
                        planned_entry_price = self.data.Close[-1]
                        sl = abs(planned_entry_price - self.get_highest_high())
                    else :
                        planned_entry_price = valid_resistant_level + self.key_level_gap
                        sl = abs(planned_entry_price - self.get_highest_high())
    
                    sl1 = planned_entry_price + sl
                    stoploss_info = self.get_stoploss_info(KeyLevel.RESISTANT, valid_resistant_level)
                    if stoploss_info is None or sl1 > stoploss_info[2]:
                        tp = self.TPcoef * sl
                        position_size = int((self.risk_percent * self.equity) / sl)
                        tp1 = planned_entry_price - tp

                        self.valid_resistant_level = valid_resistant_level
                        self.setup_duration = len(self.data.index) + self.setup_duration_gap
                        self.order_placed_time = self.data.index[-1]

                        if is_entry_in_valid_zone: 
                            self.sell(sl=sl1, tp=tp1, size=position_size, tag=self.get_tag())
                            # print(f"short {self.data.index[-1]} sl = {sl1} & sgnal = {self.signal1[-1]} & valid_resistant_level = {valid_resistant_level} & entry_price = {planned_entry_price}")
                        else:
                            self.sell(sl=sl1, tp=tp1, size=position_size, stop=planned_entry_price, tag=self.get_tag())
                            # print(f"short {self.data.index[-1]} sl = {sl1} & sgnal = {self.signal1[-1]} & valid_resistant_level = {valid_resistant_level} & planned_entry_price = {planned_entry_price}")

            if self.data['TotalSignal'][-1]==2:
                valid_support_list = Util.get_valid_support_level(self.dfopt, index, breakout_candles_lookback)
                if len(valid_support_list) > 0:
                    valid_support_level = min(valid_support_list)
                    is_entry_in_valid_zone = self.is_entry_in_valid_long_zone(valid_support_list)
                    planned_entry_price = 0
                    sl = 0
                    
                    if is_entry_in_valid_zone: 
                        planned_entry_price = self.data.Close[-1]
                        sl = abs(planned_entry_price - self.get_lowest_low())
                    else :
                        planned_entry_price = valid_support_level - self.key_level_gap
                        sl = abs(planned_entry_price - self.get_lowest_low())

                    sl1 = planned_entry_price - sl
                    stoploss_info = self.get_stoploss_info(KeyLevel.SUPPORT, valid_support_level)
                    # print(f"{stoploss_info} {valid_support_level}")
                    if stoploss_info is None or sl1 < stoploss_info[2]:
                        tp = self.TPcoef * sl
                        position_size = int((self.risk_percent * self.equity) / sl)
                        tp1 = planned_entry_price + tp

                        self.valid_support_level = valid_support_level
                        self.setup_duration = len(self.data.index) + self.setup_duration_gap
                        self.order_placed_time = self.data.index[-1]

                        if is_entry_in_valid_zone: 
                            self.buy(sl=sl1, tp=tp1, size=position_size, tag=self.get_tag())
                            # print(f"long {self.data.index[-1]} sl = {sl1} & sgnal = {self.signal1[-1]} & valid_support_level = {valid_support_level} & entry_price = {planned_entry_price}")
                        else:
                            self.buy(sl=sl1, tp=tp1, size=position_size, stop=planned_entry_price, tag=self.get_tag())
                            # print(f"long {self.data.index[-1]} sl = {sl1} & sgnal = {self.signal1[-1]} & valid_support_level = {valid_support_level} & planned_entry_price = {planned_entry_price}")
                            

In [16]:
df_m15 = df_m15_raw.loc[start_date - timedelta(hours=3): end_date]

df_h4_copy = df_h4.copy()

resistants_list = Util.define_resistant_level_with_atr(df_h4_copy, define_key_level_window, atr_multiplier=atr_multiplier, min_touched=min_touched)
supports_list = Util.define_support_level_with_atr(df_h4_copy, define_key_level_window, atr_multiplier=atr_multiplier, min_touched=min_touched)

df_h4_copy['support_point'] = df_h4_copy.apply(lambda row: Util.support_pos(supports_list, row), axis=1)
df_h4_copy['is_support'] = df_h4_copy.apply(lambda row: Util.is_support(supports_list, row), axis=1)

df_h4_copy['resistant_point'] = df_h4_copy.apply(lambda row: Util.resistant_pos(resistants_list, row), axis=1)
df_h4_copy['is_resistant'] = df_h4_copy.apply(lambda row: Util.is_resistant(resistants_list, row), axis=1)

df_h4_copy['valid_breakout_support_levels'] = Util.is_candles_above_support_level_list(df_h4_copy, breakout_candles, key_level_lookback_candle)

df_h4_copy['valid_breakout_resistant_levels'] = Util.is_candles_below_resistant_level_list(df_h4_copy, breakout_candles, key_level_lookback_candle)

df_m15_copy = df_m15.copy()
df_m15_copy = df_m15_copy.loc[start_date : end_date]
df_m15_copy = process_m15_data(df_m15_copy, df_h4_copy)

df_m15_copy['TotalSignal'], df_m15_copy['valid_resistant_levels'], df_m15_copy['valid_support_levels'] = apply_total_signal(df=df_m15_copy, key_level_lookback_candle=key_level_lookback_candle, gap=gap_look_ahead,
                                                                            breakout_candles_lookback=breakout_candles_lookback, start_trade_hour=start_trade_hour, stop_trade_hour=stop_trade_hour)

2021-01-03 23:00:00 0
2021-04-23 02:45:00 7515
2021-04-23 19:00:00 7580
2021-04-23 19:15:00 7581
2021-04-23 19:30:00 7582
2021-04-23 19:45:00 7583
2021-04-23 20:00:00 7584
2021-04-23 20:15:00 7585
2021-04-23 20:30:00 7586
2021-04-23 20:45:00 7587
2021-04-23 21:00:00 7588
2021-04-23 21:15:00 7589
2021-04-23 21:30:00 7590
2021-04-23 21:45:00 7591
2021-04-26 00:00:00 7592
2021-04-26 19:00:00 7668
2021-04-26 19:15:00 7669
2021-04-26 19:30:00 7670
2021-04-26 19:45:00 7671
2021-04-26 20:00:00 7672
2021-04-26 20:15:00 7673
2021-04-26 20:30:00 7674
2021-04-26 20:45:00 7675
2021-04-26 21:00:00 7676
2021-04-26 21:15:00 7677
2021-04-26 21:30:00 7678
2021-04-26 21:45:00 7679
2021-04-26 22:00:00 7680
2021-04-26 22:15:00 7681
2021-04-26 22:30:00 7682
2021-04-26 22:45:00 7683
2021-04-26 23:00:00 7684
2021-04-26 23:15:00 7685
2021-04-26 23:30:00 7686
2021-04-26 23:45:00 7687
2021-04-27 00:00:00 7688
2021-04-27 19:00:00 7764
2021-04-27 19:15:00 7765
2021-04-27 19:30:00 7766
2021-04-27 19:45:00 7767
202

In [ ]:
start_date_chart = '2021-1-1 00:00'
end_date_chart = '2026-9-16 00:00'
global_dfopt = df_m15_copy.loc[start_date_chart : end_date_chart]

# bt = Backtest(dfopt, MyStrat, cash=10000, margin=0.02, commission=0.0002)
bt = Backtest(global_dfopt, MyStrat, cash=10000, margin=0.02) #0.0002
stats = bt.run(dfopt=global_dfopt)
print(stats)

bt.plot()

Start                     2024-01-01 22:00:00
End                       2026-09-16 00:00:00
Duration                    988 days 02:00:00
Exposure Time [%]                     8.90653
Equity Final [$]                    12269.952
Equity Peak [$]                     12269.952
Return [%]                           22.69952
Buy & Hold Return [%]                 16.6635
Return (Ann.) [%]                     6.98985
Volatility (Ann.) [%]                 4.99639
CAGR [%]                              7.85523
Sharpe Ratio                          1.39898
Sortino Ratio                         2.62882
Calmar Ratio                          1.16082
Alpha [%]                            21.65234
Beta                                  0.06284
Max. Drawdown [%]                    -6.02149
Avg. Drawdown [%]                    -0.37109
Max. Drawdown Duration      262 days 17:15:00
Avg. Drawdown Duration        3 days 23:00:00
# Trades                                   31
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '4h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "


GridPlot(id='p1261', ...)

In [18]:
df_m15 = df_m15_raw.loc[start_date - timedelta(hours=3): end_date]

def test_multiple():
    atr_miltiplier_list = [0.6,0.7,0.8]
    min_touched_list = [3,4,5]

    with open(f"../results/result_{pair}_{time.time()}.csv", "a") as f:
        f.write(f"atr_miltiplier,min_touched,Return,ReturnYearly,MDrawdown,WinRate,NumTrades,SQN,PF\n")
        for atr_miltiplier in atr_miltiplier_list:
            for min_touched in min_touched_list:
                df_h4_copy = df_h4.copy()
                
                resistants_list = Util.define_resistant_level_with_atr(df_h4_copy, define_key_level_window, atr_multiplier=atr_miltiplier, min_touched=min_touched)
                supports_list = Util.define_support_level_with_atr(df_h4_copy, define_key_level_window, atr_multiplier=atr_miltiplier, min_touched=min_touched)

                df_h4_copy['support_point'] = df_h4_copy.apply(lambda row: Util.support_pos(supports_list, row), axis=1)
                df_h4_copy['is_support'] = df_h4_copy.apply(lambda row: Util.is_support(supports_list, row), axis=1)

                df_h4_copy['resistant_point'] = df_h4_copy.apply(lambda row: Util.resistant_pos(resistants_list, row), axis=1)
                df_h4_copy['is_resistant'] = df_h4_copy.apply(lambda row: Util.is_resistant(resistants_list, row), axis=1)

                df_h4_copy['valid_breakout_support_levels'] = Util.is_candles_above_support_level_list(df_h4_copy, breakout_candles, key_level_lookback_candle)

                df_h4_copy['valid_breakout_resistant_levels'] = Util.is_candles_below_resistant_level_list(df_h4_copy, breakout_candles, key_level_lookback_candle)

                df_m15_copy = df_m15.copy()
                df_m15_copy = df_m15_copy.loc[start_date : end_date]
                df_m15_copy = process_m15_data(df_m15_copy, df_h4_copy)

                df_m15_copy['TotalSignal'], df_m15_copy['valid_resistant_levels'], df_m15_copy['valid_support_levels'] = apply_total_signal(df=df_m15_copy, key_level_lookback_candle=key_level_lookback_candle, gap=gap_look_ahead,
                                                                                            breakout_candles_lookback=breakout_candles_lookback, start_trade_hour=start_trade_hour, stop_trade_hour=stop_trade_hour)

                global_dfopt = df_m15_copy.loc[start_date : end_date]

                # bt = Backtest(global_dfopt, MyStrat, cash=10000, margin=0.02, commission=0.0002)
                bt = Backtest(global_dfopt, MyStrat, cash=10000, margin=0.02) #0.0002
                stats = bt.run(dfopt=global_dfopt)

                f.write(f"{atr_miltiplier},{min_touched},{stats['Return [%]']},{stats['Return (Ann.) [%]']},{stats['Max. Drawdown [%]']},{stats['Win Rate [%]']},{stats['# Trades']},{stats['SQN']}, {stats['Profit Factor']}\n")
                                    
                print(f"atr_miltiplier = {atr_miltiplier} & min_touched = {min_touched} DONE")

# test_multiple()

In [19]:
df_m15 = df_m15_raw.loc[start_date - timedelta(hours=3): end_date]

def test_multiple_2():
    breakout_candles_list = [6,9,12]
    key_level_lookback_candle_list = [70,100,130]

    with open(f"../results/result_{pair}_{time.time()}.csv", "a") as f:
        f.write(f"breakout_candles,key_level_lookback_candle,Return,ReturnYearly,MDrawdown,WinRate,NumTrades,SQN,PF\n")
        for breakout_candles in breakout_candles_list:
            for key_level_lookback_candle in key_level_lookback_candle_list:
                df_h4_copy = df_h4.copy()
                
                resistants_list = Util.define_resistant_level_with_atr(df_h4_copy, define_key_level_window, atr_multiplier=atr_multiplier, min_touched=min_touched)
                supports_list = Util.define_support_level_with_atr(df_h4_copy, define_key_level_window, atr_multiplier=atr_multiplier, min_touched=min_touched)

                df_h4_copy['support_point'] = df_h4_copy.apply(lambda row: Util.support_pos(supports_list, row), axis=1)
                df_h4_copy['is_support'] = df_h4_copy.apply(lambda row: Util.is_support(supports_list, row), axis=1)

                df_h4_copy['resistant_point'] = df_h4_copy.apply(lambda row: Util.resistant_pos(resistants_list, row), axis=1)
                df_h4_copy['is_resistant'] = df_h4_copy.apply(lambda row: Util.is_resistant(resistants_list, row), axis=1)

                df_h4_copy['valid_breakout_support_levels'] = Util.is_candles_above_support_level_list(df_h4_copy, breakout_candles, key_level_lookback_candle)

                df_h4_copy['valid_breakout_resistant_levels'] = Util.is_candles_below_resistant_level_list(df_h4_copy, breakout_candles, key_level_lookback_candle)

                df_m15_copy = df_m15.copy()
                df_m15_copy = df_m15_copy.loc[start_date : end_date]
                df_m15_copy = process_m15_data(df_m15_copy, df_h4_copy)

                df_m15_copy['TotalSignal'], df_m15_copy['valid_resistant_levels'], df_m15_copy['valid_support_levels'] = apply_total_signal(df=df_m15_copy, key_level_lookback_candle=key_level_lookback_candle, gap=gap_look_ahead,
                                                                                            breakout_candles_lookback=breakout_candles_lookback, start_trade_hour=start_trade_hour, stop_trade_hour=stop_trade_hour)

                global_dfopt = df_m15_copy.loc[start_date : end_date]

                # bt = Backtest(global_dfopt, MyStrat, cash=10000, margin=0.02, commission=0.0002)
                bt = Backtest(global_dfopt, MyStrat, cash=10000, margin=0.02) #0.0002
                stats = bt.run(dfopt=global_dfopt)

                f.write(f"{breakout_candles},{key_level_lookback_candle},{stats['Return [%]']},{stats['Return (Ann.) [%]']},{stats['Max. Drawdown [%]']},{stats['Win Rate [%]']},{stats['# Trades']},{stats['SQN']}, {stats['Profit Factor']}\n")
                                    
                print(f"breakout_candles = {breakout_candles} & key_level_lookback_candle = {key_level_lookback_candle} DONE")

# test_multiple_2()